<a href="https://colab.research.google.com/github/neto995/data-science-tripleten/blob/main/megaline_classification_portfolio_en.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Mental Map: Training and Evaluation Flow

```text
users_behavior.csv
        ↓
1. Understand the data
        ↓
2. Define features and target
        ↓
3. Split the data
   ├── TRAIN
   ├── VALIDATION
   └── TEST
        ↓
4. Train multiple models
   ├── Decision Tree
   ├── Random Forest
   └── Logistic Regression
        ↓
5. Tune hyperparameters
        ↓
6. Compare Accuracy on VALIDATION
        ↓
7. Choose ONE model
        ↓
8. Evaluate it ONE time on TEST
        ↓
9. Perform a sanity check
        ↓
10. Final conclusions
```

## Diagram: Train, Validation, and Test

```text
100% DATASET
      │
      ├── TRAIN
      │      ↓
      │    fit()
      │      ↓
      │   LEARNS
      │
      ├── VALIDATION
      │      ↓
      │   predict()
      │      ↓
      │ choose model
      │
      └── TEST
             ↓
          predict()
             ↓
       FINAL evaluation
```

# Step 1: Understanding the Data

In [28]:
import pandas as pd

# Load the dataset from Google Drive
df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/Proyecto 10/users_behavior.csv")


df.head()

,calls,minutes,messages,mb_used,is_ultra
0,40.0,311.90,83.0,19915.42,0
1,85.0,516.75,56.0,22696.96,0
2,77.0,467.66,86.0,21060.45,0
3,106.0,745.53,81.0,8437.39,1
4,66.0,418.74,1.0,14502.75,0


In [29]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3214 entries, 0 to 3213
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   calls     3214 non-null   float64
 1   minutes   3214 non-null   float64
 2   messages  3214 non-null   float64
 3   mb_used   3214 non-null   float64
 4   is_ultra  3214 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 125.7 KB


In [30]:
df.describe()

,calls,minutes,messages,mb_used,is_ultra
count,3214.000000,3214.000000,3214.000000,3214.000000,3214.000000
mean,63.038892,438.208787,38.281269,17207.673836,0.306472
std,33.236368,234.569872,36.148326,7570.968246,0.461100
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,40.000000,274.575000,9.000000,12491.902500,0.000000
50%,62.000000,430.600000,30.000000,16943.235000,0.000000
75%,82.000000,571.927500,57.000000,21424.700000,1.000000
max,244.000000,1632.060000,224.000000,49745.730000,1.000000


In [31]:
df['is_ultra'].value_counts()

,count
is_ultra,
0,2229
1,985


In [32]:
df['is_ultra'].value_counts(normalize=True)

,proportion
is_ultra,
0,0.693528
1,0.306472


### Understanding the Data: Defining the Sanity Check

I don't have any null values.

The target variable `is_ultra` has the following distribution:

- 2,229 Smart → **69.35%**
- 985 Ultra → **30.64%**

Since **Smart is the majority class**, a very basic strategy would be to predict **Smart for every customer**.

Without learning any patterns, this strategy would achieve approximately **69.35% accuracy** on the full dataset.

Therefore:

> **69.35% is my initial baseline for the sanity check.**

If my model performs above this baseline, it gives me evidence that it is learning useful patterns from customer behavior instead of simply taking advantage of Smart being the majority class.

However, just beating the baseline is not enough for this project. The requirement is to achieve at least **75% accuracy**.

```text
Initial baseline  → 69.35%
Project minimum   → 75%
Final model       → target > 75%
```

# Step 2: Define Features and Target

In [33]:
features = df.drop(['is_ultra'], axis=1)
target = df['is_ultra']

I use all columns as **features**, except `is_ultra`, which is my **target**.

The target contains the answer I want the model to predict:

- `0` → Smart
- `1` → Ultra

# Step 3: Split the Data

### Dataset Split Logic

**Train (60%)** → I use 60% of the dataset to **train the model**. This is where the model observes the data and learns patterns from the features and target.

**Validation (20%)** → I give the model the **features without giving it the answers**. The model makes predictions with `predict()`, and I compare them against `target_valid`. I use this dataset to decide **which model and hyperparameters perform best**.

**Test (20%)** → Once I have selected the model and its hyperparameters, I give it the final 20% of the features, which were **not used for training or model selection**. I compare its predictions against `target_test`.

I think of this as the model's **final exam**.

```text
100% DATASET
│
├── 60% TRAIN
│      features + target
│           ↓
│         fit()
│           ↓
│        LEARNS
│
├── 20% VALIDATION
│      features → predict()
│           ↓
│      vs. target_valid
│           ↓
│   CHOOSE MODEL + HYPERPARAMETERS
│
└── 20% TEST
       features → predict()
            ↓
       vs. target_test
            ↓
       FINAL ACCURACY
```

### Model Competition

```text
                    60% TRAIN
          features_train + target_train
                       │
          ┌────────────┼────────────┐
          ↓            ↓            ↓
    Decision Tree  Random Forest  Logistic Reg.
          ↓            ↓            ↓
        fit()         fit()         fit()
          │            │            │
          └────────────┼────────────┘
                       ↓
                20% VALIDATION
                       ↓
                Compare Accuracy
                       ↓
                  WHO WON?
```

### Split

First, I separate 40% of the dataset and store it temporarily as `rest`.

Then, I split that 40% in half:

- 20% → Validation
- 20% → Test

The remaining 60% becomes my Training set.

After creating the three datasets, I can start training and comparing my models.

In [34]:
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

# Keep 60% for training and store the remaining 40% as "rest"
features_train, features_rest, target_train, target_rest = train_test_split(
    features,
    target,
    test_size=0.4,
    random_state=54321
)

# Split the remaining 40% into two equal parts:
# 20% validation and 20% test
features_valid, features_test, target_valid, target_test = train_test_split(
    features_rest,
    target_rest,
    test_size=0.5,
    random_state=54321
)

# Step 4: Model Competition

I first compare the three models using their **default parameters**, without tuning hyperparameters.

This gives me a starting point to understand how each algorithm performs before making adjustments.

## Decision Tree Classifier

In [35]:
from sklearn.tree import DecisionTreeClassifier

# STEP 1: TRAIN MODEL

# 1.1 Create the model
model = DecisionTreeClassifier(
    random_state=54321
)

# 1.2 Train the model using the training set
model.fit(features_train, target_train)


# STEP 2: VALIDATION

# 2.1 Predict using what the model learned,
# without giving it the correct answers
predictions_valid = model.predict(features_valid)

# 2.2 Compare the predictions against the real answers
accuracy = accuracy_score(
    target_valid,
    predictions_valid
)

print("Decision Tree validation accuracy:", accuracy)

Decision Tree validation accuracy: 0.687402799377916


## Random Forest Classifier

In [36]:
from sklearn.ensemble import RandomForestClassifier

# STEP 1: TRAIN MODEL

# 1.1 Create the model
model = RandomForestClassifier(
    random_state=54321
)

# 1.2 Train the model using the training set
model.fit(features_train, target_train)


# STEP 2: VALIDATION

# 2.1 Predict using what the model learned,
# without giving it the correct answers
predictions_valid = model.predict(features_valid)

# 2.2 Compare the predictions against the real answers
accuracy = accuracy_score(
    target_valid,
    predictions_valid
)

print("Random Forest validation accuracy:", accuracy)

Random Forest validation accuracy: 0.7822706065318819


## Logistic Regression

In [37]:
from sklearn.linear_model import LogisticRegression

# STEP 1: TRAIN MODEL

# 1.1 Create the model
model = LogisticRegression(
    random_state=54321
)

# 1.2 Train the model using the training set
model.fit(features_train, target_train)


# STEP 2: VALIDATION

# 2.1 Predict using what the model learned,
# without giving it the correct answers
predictions_valid = model.predict(features_valid)

# 2.2 Compare the predictions against the real answers
accuracy = accuracy_score(
    target_valid,
    predictions_valid
)

print("Logistic Regression validation accuracy:", accuracy)

Logistic Regression validation accuracy: 0.7076205287713841


## Summary

Results using default parameters, before hyperparameter tuning.

| Model | Validation Accuracy |
|---|---:|
| **Decision Tree** | 68.74% |
| **Random Forest** | **78.23%** |
| **Logistic Regression** | 70.76% |

At this point, **Random Forest** is the best-performing model with a validation accuracy of **78.23%**, already exceeding the project's minimum requirement of 75%.

# Step 5: Model Competition with Hyperparameter Tuning

Now I adjust some of the models' **hyperparameters** to see whether I can improve their performance.

Instead of accepting the default configuration, I test different values and compare their accuracy using the same Validation set.

## Decision Tree Classifier

In [38]:
best_accuracy = 0
best_depth = 0

for depth in range(1, 10):

    model = DecisionTreeClassifier(
        random_state=54321,
        max_depth=depth
    )

    model.fit(features_train, target_train)

    predictions_valid = model.predict(features_valid)

    accuracy = accuracy_score(
        target_valid,
        predictions_valid
    )

    print("max_depth:", depth, "accuracy:", accuracy)

    if accuracy > best_accuracy:
        best_accuracy = accuracy
        best_depth = depth

print("Best max_depth:", best_depth)
print("Best accuracy:", best_accuracy)

max_depth: 1 accuracy: 0.7216174183514774
max_depth: 2 accuracy: 0.7418351477449455
max_depth: 3 accuracy: 0.7651632970451011
max_depth: 4 accuracy: 0.744945567651633
max_depth: 5 accuracy: 0.7651632970451011
max_depth: 6 accuracy: 0.7542768273716952
max_depth: 7 accuracy: 0.7433903576982893
max_depth: 8 accuracy: 0.7511664074650077
max_depth: 9 accuracy: 0.7682737169517885
Best max_depth: 9
Best accuracy: 0.7682737169517885


## Random Forest Classifier

In [20]:
best_accuracy = 0
best_est = 0
best_depth = 0

for est in range(10, 51, 10):
    for depth in range(1, 11):

        model = RandomForestClassifier(
            random_state=54321,
            n_estimators=est,
            max_depth=depth
        )

        model.fit(features_train, target_train)

        predictions_valid = model.predict(features_valid)

        accuracy = accuracy_score(
            target_valid,
            predictions_valid
        )

        print(
            "n_estimators:", est,
            "max_depth:", depth,
            "accuracy:", accuracy
        )

        if accuracy > best_accuracy:
            best_accuracy = accuracy
            best_est = est
            best_depth = depth

print("Best n_estimators:", best_est)
print("Best max_depth:", best_depth)
print("Best accuracy:", best_accuracy)

n_estimators: 10 max_depth: 1 accuracy: 0.71850699844479
n_estimators: 10 max_depth: 2 accuracy: 0.7465007776049767
n_estimators: 10 max_depth: 3 accuracy: 0.7480559875583204
n_estimators: 10 max_depth: 4 accuracy: 0.7853810264385692
n_estimators: 10 max_depth: 5 accuracy: 0.7807153965785381
n_estimators: 10 max_depth: 6 accuracy: 0.7838258164852255
n_estimators: 10 max_depth: 7 accuracy: 0.7884914463452566
n_estimators: 10 max_depth: 8 accuracy: 0.7993779160186625
n_estimators: 10 max_depth: 9 accuracy: 0.7869362363919129
n_estimators: 10 max_depth: 10 accuracy: 0.80248833592535
n_estimators: 20 max_depth: 1 accuracy: 0.702954898911353
n_estimators: 20 max_depth: 2 accuracy: 0.7293934681181959
n_estimators: 20 max_depth: 3 accuracy: 0.7713841368584758
n_estimators: 20 max_depth: 4 accuracy: 0.7807153965785381
n_estimators: 20 max_depth: 5 accuracy: 0.7791601866251944
n_estimators: 20 max_depth: 6 accuracy: 0.7791601866251944
n_estimators: 20 max_depth: 7 accuracy: 0.7822706065318819
n

## Logistic Regression

`C` controls the regularization strength of Logistic Regression.

I'm testing several values to see whether changing the regularization improves the model's validation accuracy.

In [21]:
best_accuracy = 0
best_c = 0

for c in [0.01, 0.1, 1, 10, 100]:

    model = LogisticRegression(
        random_state=54321,
        solver="liblinear",
        C=c
    )

    model.fit(features_train, target_train)

    predictions_valid = model.predict(features_valid)

    accuracy = accuracy_score(
        target_valid,
        predictions_valid
    )

    print("C:", c, "accuracy:", accuracy)

    if accuracy > best_accuracy:
        best_accuracy = accuracy
        best_c = c

print("Best C:", best_c)
print("Best accuracy:", best_accuracy)

C: 0.01 accuracy: 0.6734059097978227
C: 0.1 accuracy: 0.6780715396578538
C: 1 accuracy: 0.6780715396578538
C: 10 accuracy: 0.6780715396578538
C: 100 accuracy: 0.6780715396578538
Best C: 0.1
Best accuracy: 0.6780715396578538


The tested values produced very similar accuracy scores.

In this case, tuning `C` did not meaningfully improve the performance of Logistic Regression.

# Step 6: Compare Accuracy on VALIDATION

## Model Competition After Hyperparameter Tuning

| Model | Default Accuracy | Best Configuration | Tuned Accuracy |
|---|---:|---|---:|
| Decision Tree | 68.74% | `max_depth=9` | **76.83%** |
| Random Forest | **78.23%** | `n_estimators=10`, `max_depth=10` | **80.25%** 🏆 |
| Logistic Regression | 70.76% | `C=0.1` | **67.81%** |

The important result here is that **Random Forest remains the strongest model after tuning**, reaching **80.25% validation accuracy**.

# Step 7: Choose One Model

### Summary

After comparing the models on the Validation set:

- **Decision Tree:** improved after tuning `max_depth`.
- **Random Forest:** improved from **78.23% → 80.25%** and achieved the best validation result. 🏆
- **Logistic Regression:** remained below Random Forest.
- **Winner on Validation:** `RandomForestClassifier`.
- **Winning configuration:** `n_estimators=10`, `max_depth=10`.
- **Next step:** evaluate the winning model using the untouched **Test set**.

At this point, I stop comparing models. The Test set will now be used for the final evaluation.

# Step 8: Evaluate the Final Model on TEST

In [22]:
# STEP 3: TEST

# 3.1 Create the winning model with the best hyperparameters
final_model = RandomForestClassifier(
    random_state=54321,
    n_estimators=10,
    max_depth=10
)

# 3.2 Train the model
final_model.fit(features_train, target_train)

# 3.3 The model takes its "final exam" using features
# that were not used during training or model selection
predictions_test = final_model.predict(features_test)

# 3.4 Compare its predictions against the real answers
accuracy_test = accuracy_score(
    target_test,
    predictions_test
)

print("Final Test accuracy:", accuracy_test)

Final Test accuracy: 0.8242612752721618


### Test Conclusion

The selected model is a `RandomForestClassifier` with:

- `n_estimators = 10`
- `max_depth = 10`
- `random_state = 54321`

The model achieved **82.43% accuracy on the Test set**, exceeding:

- **Initial baseline:** 69.35%
- **Minimum required accuracy:** 75%
- **Validation accuracy:** 80.25%

In other words, when the model receives customers that were not used to train it or select its hyperparameters, it classifies approximately **82% of them correctly**.

**Final Test Accuracy: 82.43%**

# Step 9: Sanity Check

Did my model actually learn how to identify customer behavior, or is it simply taking advantage of the fact that most customers belong to the **Smart** plan?

To test this, I compare my Random Forest against a very simple model that always predicts the majority class.

If my trained model performed worse than this simple strategy, there would be little reason to use it. I could simply recommend Smart to everyone and still be correct most of the time.

This gives me a useful reality check for my model.

In [23]:
from sklearn.dummy import DummyClassifier

# Create a "dummy" model that always predicts the most frequent class
dummy_model = DummyClassifier(strategy="most_frequent")

# Learn which class is most frequent in TRAIN
dummy_model.fit(features_train, target_train)

# Always predict that class on TEST
dummy_predictions = dummy_model.predict(features_test)

# Calculate its accuracy
dummy_accuracy = accuracy_score(
    target_test,
    dummy_predictions
)

print("Dummy baseline accuracy:", dummy_accuracy)
print("Random Forest accuracy:", accuracy_test)

Dummy baseline accuracy: 0.7247278382581649
Random Forest accuracy: 0.8242612752721618


# Step 10: Conclusions

## Conclusions

The final predictive model appears useful because it exceeds all of the main comparison points:

1. **Initial baseline:** Smart represents approximately 69.35% of the full dataset, so always predicting Smart would already produce a relatively high accuracy without learning anything.

2. **Sanity check with `DummyClassifier`:** on the Test set, always predicting the majority class achieves **72.47% accuracy**.

3. **Project requirement:** the model needed to achieve at least **75% accuracy**.

4. **Random Forest:** the model performed well with its default configuration, and hyperparameter tuning improved its Validation performance.

5. **Final model:** `RandomForestClassifier` with `n_estimators=10` and `max_depth=10` achieved **82.43% accuracy on the Test set**.

The final model outperforms the `DummyClassifier` by approximately **10 percentage points**.

This gives me evidence that the model is not simply predicting Smart because it is the majority class.

Instead, it is learning patterns from customer behavior — including calls, minutes, messages, and mobile data usage — to make a better classification between the Smart and Ultra plans.

# Exporting the Model for the Interactive Portfolio

After training and evaluating the model in Python, I wanted to use the same trained model in my web portfolio.

For this, I exported the Random Forest model to **ONNX (Open Neural Network Exchange)**.

ONNX is not converting my Python code into JavaScript.

Instead, it stores the **trained model** in a standardized format that can be executed outside the Python environment.

In my case:

```text
Python / scikit-learn
        ↓
Train Random Forest
        ↓
Export trained model
        ↓
ONNX (.onnx)
        ↓
ONNX Runtime Web
        ↓
Interactive prediction in my portfolio
```

This allows me to keep Python and scikit-learn as the Data Science layer while using the trained model inside the browser for the interactive demo.

In [24]:
!pip install skl2onnx onnx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.2/317.2 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 81.1 MB/s eta 0:00:00


In [25]:
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType

# The model expects 4 input features:
# calls, minutes, messages, mb_used
initial_type = [
    ("input", FloatTensorType([None, 4]))
]

# Convert the trained scikit-learn model to ONNX.
# zipmap=False keeps probabilities as numeric tensor output,
# which is easier to use with ONNX Runtime Web.
onnx_model = convert_sklearn(
    final_model,
    initial_types=initial_type,
    options={
        id(final_model): {
            "zipmap": False
        }
    }
)

# Save the exported model
with open("megaline_random_forest.onnx", "wb") as file:
    file.write(onnx_model.SerializeToString())

### Download the ONNX Model

After converting the trained Random Forest, I save the model as:

`megaline_random_forest.onnx`

This is the file that I later use in the interactive portfolio.

In [26]:
# Uncomment these lines in Google Colab if I want
# to download the ONNX file to my computer.

# from google.colab import files
# files.download("megaline_random_forest.onnx")

## Deployment Validation

Before using the ONNX model in the portfolio, I created a few manual test cases in Python.

The purpose is simple: I want reference predictions from the original scikit-learn model so I can compare them against the predictions produced by the ONNX model in the browser.

```text
Python Random Forest
        ↓
same customer inputs
        ↓
expected predictions

ONNX model in browser
        ↓
same customer inputs
        ↓
compare results
```

If both versions return consistent predictions, I have more confidence that the model was exported and integrated correctly.

In [27]:
# Reference cases for deployment validation
test_cases = pd.DataFrame([
    {
        "calls": 0,
        "minutes": 0,
        "messages": 0,
        "mb_used": 0
    },
    {
        "calls": 40,
        "minutes": 130,
        "messages": 80,
        "mb_used": 20000
    },
    {
        "calls": 250,
        "minutes": 1500,
        "messages": 250,
        "mb_used": 50000
    }
])

print("Predicted classes:")
print(final_model.predict(test_cases))

print("\nClass probabilities:")
print(final_model.predict_proba(test_cases))

Predicted classes:
[1 0 1]

Class probabilities:
[[0.28333333 0.71666667]
 [0.55086894 0.44913106]
 [0.1        0.9       ]]
